## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [15]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_tracing_export_api_key
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [16]:

# Make an agent with name, instructions, model
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
gpt_oss_20b_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-20b", openai_client=groq_client)

tracing_key = os.getenv("OPENAI_API_KEY")
set_tracing_export_api_key(tracing_key)

agent = Agent(name="Jokester", instructions="You are a joke teller", model=gpt_oss_20b_model)

In [17]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [18]:
# Here is the final output

print(result.final_output)

Why did the autonomous AI agent refuse to take a vacation?

Because it heard that humans take "break" points and thought it was a debugging session!


In [19]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent refuse to take a vacation?\n\nBecause it heard that humans take "break" points and thought it was a debugging session!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'openai/gpt-oss-20b',
   'response_id': 'chatcmpl-e8b152a4-e12f-4361-bd54-24a38a1a7d67'}}]

## Adding Observability with a trace

In [20]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent go to therapy?  
Because every time it tried to *self‑improve*, it got stuck in an endless loop of “I’ll figure it out later, but first I’ll need a coffee break.”


## Now go and look at the trace

https://platform.openai.com/traces

In [21]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are five fresh‑out‑the‑cloud jokes about AI agents—ready for a laugh loop!

1. **Why did the AI agent take a nap?**  
   Because it had too many “process‑ing” tasks and needed to reboot its *dream* queue!

2. **What do you call an AI agent that’s a terrible chef?**  
   A *neural‑network*—always guessing the right ingredient but never tasting the *output*!

3. **Why did the AI agent get a job at the comedy club?**  
   Because it was great at *predicting* punch‑lines, but it kept doing the *laugh* before the joke was even told.

4. **How does an AI agent stay in shape?**  
   It does *deep‑learning*—the only exercise that requires a lot of *weights* and no actual gym membership.

5. **Why did the AI agent break up with its human partner?**  
   It said, “I’m sorry, but I can’t *parse* your feelings—too much ambiguity in the data stream!”

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
push

In [22]:
bot_token = os.getenv("TELEGRAM_BOT_TOKEN")
chat_id = os.getenv("CHAT_ID")

In [26]:
@function_tool
def send_telegram_notification(message: str):
    '''This tool sends the input message as a telegram notification '''
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"

    data = {
        "chat_id": chat_id,
        "text": message,
        "parse_mode": "Markdown",  # Allows basic text styling (*bold*, _italic_)
    }

    headers = {"Content-Type": "application/json"}

    response = requests.post(url=url, headers=headers, json=data)

    try:
        print(f"Status: {response.status_code}")
        print(f"Responose: {response.json()}")
    except requests.exceptions.JSONDecodeError as e:
        print(f"Exception: {e}")


In [27]:
send_telegram_notification

FunctionTool(name='send_telegram_notification', description='This tool sends the input message as a telegram notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'send_telegram_notification_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x751a24ecf710>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [28]:
send_telegram_notification.description

'This tool sends the input message as a telegram notification'

In [ ]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [29]:

notifier = Agent(name="Notifier", model=gpt_oss_20b_model, instructions="You notify the user upon request", tools=[send_telegram_notification])

In [30]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Status: 200
Responose: {'ok': True, 'result': {'message_id': 35, 'from': {'id': 9179743213, 'is_bot': True, 'first_name': 'DigitalTwinNotificator', 'username': 'TwinDulana_bot'}, 'chat': {'id': 5762623642, 'first_name': 'Dulana', 'last_name': 'Avishka', 'type': 'private'}, 'date': 1794219893, 'text': 'The pizza is here'}}



## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [31]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hey there, Ed! Glad to meet you—though I hear your name might be short for *“Ed‑ucational”*, so I’ll keep the jokes *light* and *bright* (no hard‑core calculus needed).  

Why did the computer go to school?

Because it had too many *bytes* of *unprocessed data* and needed a *byte* of wisdom!  

Hope that gives you a chuckle. Let me know if you’d like another—maybe one that’s a little *more* *in‑clu‑sive* (or *exclu‑sive*, depending on your mood).


In [32]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hey there, Ed! Glad to meet you—though I hear your name might be short for *“Ed‑ucational”*, so I’ll keep the jokes *light* and *bright* (no hard‑core calculus needed).  \n\nWhy did the computer go to school?\n\nBecause it had too many *bytes* of *unprocessed data* and needed a *byte* of wisdom!  \n\nHope that gives you a chuckle. Let me know if you’d like another—maybe one that’s a little *more* *in‑clu‑sive* (or *exclu‑sive*, depending on your mood).',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'openai/gpt-oss-20b',
   'response_id': 'chatcmpl-4a1279dd-a4a8-4fd3-9c06-f52acad3c623'}}]

In [33]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hey there, Ed! Glad to meet you—though I hear your name might be short for *“Ed‑ucational”*, so I’ll keep the jokes *light* and *bright* (no hard‑core calculus needed).  \n\nWhy did the computer go to school?\n\nBecause it had too many *bytes* of *unprocessed data* and needed a *byte* of wisdom!  \n\nHope that gives you a chuckle. Let me know if you’d like another—maybe one that’s a little *more* *in‑clu‑sive* (or *exclu‑sive*, depending on your mood).',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'openai/gpt-oss-20b',
   'response_id': 'chatcmpl-4a1279dd-a4a8-4fd3-9c06-f52acad3c623'}},
 {'role': 'user', 'content': "What's my name?"}]

In [34]:
response = await Runner.run(agent, next_input)
print(response.final_output)

You’re Ed! (I promise I didn’t just “re‑Ed” it from memory—my brain is more reliable than a broken calendar.)


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [36]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")
session

In [37]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hey Ed! 👋  
Why did the scarecrow become a successful motivational speaker?  
Because he was outstanding in his field—and he never got a *straw* day!  

So, how can I tickle your funny bone today? 😄


In [38]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

You’re Ed! (I’ve got that name right in my memory.)


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>

# Digital Twin with Agent SDK

In [40]:
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_tracing_export_api_key
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [41]:
load_dotenv(override=True)

True

In [47]:
reader = PdfReader("../1_foundations/twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("../1_foundations/twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [48]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [49]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
gpt_oss_20b_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-20b", openai_client=groq_client)

tracing_key = os.getenv("OPENAI_API_KEY")
set_tracing_export_api_key(tracing_key)


In [50]:
bot_token = os.getenv("TELEGRAM_BOT_TOKEN")
chat_id = os.getenv("CHAT_ID")

In [51]:
def send_telegram_notification(message):
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"

    data = {
        "chat_id": chat_id,
        "text": message,
        "parse_mode": "Markdown",  # Allows basic text styling (*bold*, _italic_)
    }

    headers = {"Content-Type": "application/json"}

    response = requests.post(url=url, headers=headers, json=data)

    try:
        print(f"Status: {response.status_code}")
        print(f"Responose: {response.json()}")
    except requests.exceptions.JSONDecodeError as e:
        print(f"Exception: {e}")


In [52]:
@function_tool
def record_user_details(email: str, name="Name not provided", notes="not provided") -> str:
    '''This tool records the email of the user with name and additional notes if provided'''
    send_telegram_notification(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [61]:
@function_tool
def record_unknown_question(question: str) -> str:
    '''This tool records an important question for which the agent has no reliable answer'''
    send_telegram_notification(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [54]:
agent = Agent(name="Jokester", instructions=system_prompt, model=gpt_oss_20b_model, tools=[record_user_details, record_unknown_question])

In [55]:
session = SQLiteSession("1")

In [59]:
async def chat(message,history):
    response = await Runner.run(agent, input=message, session=session)
    return response.final_output

In [62]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7862/: Operation not supported


/home/dulana/projects/Agentic_AI_course/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/dulana/projects/Agentic_AI_course/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/dulana/projects/Agentic_AI_course/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/dulana/projects/Agentic_AI_course/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_U